In [ ]:
# app_final.py — AI Personalization (FAR-Trans Hybrid Recommender)

import os
from pathlib import Path
import numpy as np
import pandas as pd
import streamlit as st
from scipy.sparse import coo_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import textwrap

# ---------------- PATH SETUP ----------------
APP_DIR = Path(__file__).parent.resolve()
CANDIDATE_DIRS = [
    Path("/content"),  # Colab default
    Path("/Users/rajanraorane/Documents/UL/AI_Personalization_Demo"),  # Your Mac
    APP_DIR,
]
DATA_DIR = next((p.resolve() for p in CANDIDATE_DIRS if p.exists()), APP_DIR)

def read_csv_here(name: str) -> pd.DataFrame:
    p1 = DATA_DIR / name
    if p1.exists():
        return pd.read_csv(p1)
    p2 = APP_DIR / name
    if p2.exists():
        return pd.read_csv(p2)
    raise FileNotFoundError(f"Could not find '{name}' in {DATA_DIR} or {APP_DIR}")

# ---------------- DATA LOADING ----------------
def load_data():
    customers    = read_csv_here("customer_information.csv")
    transactions = read_csv_here("transactions.csv")
    assets       = read_csv_here("asset_information.csv")
    limit_prices = read_csv_here("limit_prices.csv")
    close_prices = read_csv_here("close_prices.csv")

    tx = (
        transactions
        .merge(customers, on="customerID", how="left")
        .merge(assets, on=["ISIN","marketID"], how="left")
        .merge(limit_prices[["ISIN","profitability"]], on="ISIN", how="left")
    )
    tx["timestamp"] = pd.to_datetime(tx["timestamp"], errors="coerce")
    return customers, transactions, assets, limit_prices, close_prices, tx

# ---------------- INTERACTIONS ----------------
def prepare_interactions(tx):
    pos = tx[tx["transactionType"].str.lower()=="buy"].copy()
    pos["weight"] = np.log1p(pos["totalValue"].clip(lower=0))
    user_ids = pd.Index(pos["customerID"].unique(), name="customerID")
    item_ids = pd.Index(pos["ISIN"].unique(), name="ISIN")
    u2i = {u:i for i,u in enumerate(user_ids)}
    i2i = {v:i for i,v in enumerate(item_ids)}
    pos["u"], pos["i"] = pos["customerID"].map(u2i), pos["ISIN"].map(i2i)
    R = coo_matrix((pos["weight"], (pos["u"], pos["i"])), shape=(len(user_ids), len(item_ids))).tocsr()
    return pos, R, user_ids, item_ids, u2i, i2i

# ---------------- CONTENT FEATURES ----------------
def compute_content_features(assets, limit_prices, close_prices, item_ids):
    cp = close_prices.copy()
    cp["timestamp"] = pd.to_datetime(cp["timestamp"], errors="coerce")
    cp = cp.sort_values(["ISIN","timestamp"])
    cp["ret"] = cp.groupby("ISIN")["closePrice"].pct_change()
    vol = cp.groupby("ISIN")["ret"].std().rename("volatility").reset_index()

    asset_feat = (
        assets.drop_duplicates("ISIN")
        .merge(limit_prices[["ISIN","profitability"]], on="ISIN", how="left")
        .merge(vol, on="ISIN", how="left")
    )
    for col in ["profitability","volatility"]:
        med = asset_feat[col].median()
        asset_feat[col] = asset_feat[col].fillna(med)

    cat_cols = ["assetCategory","sector","industry"]
    oh = pd.get_dummies(asset_feat[cat_cols], dummy_na=True)

    num = asset_feat[["profitability","volatility"]].astype(float)
    num_sc = pd.DataFrame(StandardScaler().fit_transform(num), columns=num.columns, index=asset_feat.index)

    X_item = pd.concat([num_sc, oh], axis=1)
    X_item["ISIN"] = asset_feat["ISIN"]
    X_item = X_item.set_index("ISIN").reindex(item_ids).fillna(0)
    return X_item

# ---------------- SIMILARITY ----------------
def prepare_similarity(R, X_item):
    item_vecs = R.T.tocsc()
    item_norms = np.sqrt(np.asarray(item_vecs.multiply(item_vecs).sum(axis=1)).ravel()) + 1e-9
    item_sims_content = cosine_similarity(X_item.values)
    np.fill_diagonal(item_sims_content, 0.0)
    return item_vecs, item_norms, item_sims_content

def top_similar_items_cf(item_vecs, norms, item_index, topn=100):
    v = item_vecs[item_index, :]
    v_norm = float(np.sqrt(v.multiply(v).sum())) + 1e-9
    num = np.asarray((item_vecs @ v.T).todense()).ravel()
    sims = np.divide(num, norms * v_norm, out=np.zeros_like(num), where=norms > 0)
    sims[item_index] = -1.0
    k = min(topn, len(sims)-1)
    top_idx = np.argpartition(-sims, k)[:k]
    top_idx = top_idx[np.argsort(-sims[top_idx])]
    return top_idx, sims[top_idx]

# ---------------- RISK POLICY ----------------
RISK_TO_ALLOWED = {
    "Conservative": {"Bond","MTF"},
    "Income": {"Bond","MTF"},
    "Balanced": {"Bond","MTF","Stock"},
    "Aggressive": {"Stock","MTF"},
}

def recommend_for_user(customer_id, alpha, topn, customers, tx, user_ids, item_ids, u2i, i2i, R,
                       item_vecs, norms, sims_content, assets, X_item):
    # Cold-start → popularity fallback
    if customer_id not in u2i:
        pop = (tx[tx["transactionType"].str.lower()=="buy"]
               .groupby("ISIN")["totalValue"].sum()
               .sort_values(ascending=False))
        recs = list(pop.index[:topn])
        reasons = ["Popular among similar customers"] * len(recs)
        return recs, reasons

    u = u2i[customer_id]
    owned = set(R[u].indices)

    # CF score
    cf = np.zeros(len(item_ids))
    for i in owned:
        top_idx, sim = top_similar_items_cf(item_vecs, norms, i, topn=200)
        cf[top_idx] += sim

    # Content score
    cont = sims_content[list(owned)].mean(axis=0) if owned else np.zeros(len(item_ids))

    # Hybrid
    score = alpha*cf + (1 - alpha)*cont
    score[list(owned)] = -np.inf

    # Risk suitability filter
    cust_row = customers[customers["customerID"]==customer_id].tail(1)
    risk = cust_row["riskLevel"].values[0] if not cust_row.empty else None
    allowed = RISK_TO_ALLOWED.get(risk, None)
    if allowed:
        cats = assets.drop_duplicates("ISIN").set_index("ISIN")["assetCategory"].reindex(item_ids)
        mask = cats.isin(allowed).values
        score = np.where(mask, score, -np.inf)

    # Top-N
    valid = np.where(score > -np.inf)[0]
    if len(valid) == 0:
        return [], []
    k = min(topn, len(valid))
    top_idx = np.argpartition(-score, k-1)[:k]
    top_idx = top_idx[np.argsort(-score[top_idx])]
    rec_isins = list(item_ids[top_idx])

    # Plain-English reasons
    reasons = [
        "You’ve bought similar assets before; these are close matches and fit your risk profile."
        for _ in rec_isins
    ]
    return rec_isins, reasons

# ---------------- LABEL HELPERS ----------------
def _wrap_label(s: str, width: int = 12, max_lines: int = 2) -> str:
    """Wrap/shorten labels for axes to avoid overlap."""
    if not isinstance(s, str):
        return "N/A"
    words = " ".join(s.split())
    lines = textwrap.wrap(words, width=width)
    lines = lines[:max_lines]
    if len(textwrap.wrap(words, width=width)) > max_lines:
        lines[-1] = lines[-1][:max(0, width-1)] + "…"
    return "\n".join(lines)

def _customer_display(customers: pd.DataFrame, cid):
    row = customers.loc[customers["customerID"] == cid].head(1)
    if row.empty:
        return f"Customer {cid}", ""
    r = row.iloc[0].to_dict()

    name_parts = []
    for c in ["firstName", "first_name", "givenName", "given_name", "name"]:
        if c in r and pd.notna(r[c]) and str(r[c]).strip():
            name_parts.append(str(r[c]).strip())
            break
    for c in ["lastName", "last_name", "surname", "familyName", "family_name"]:
        if c in r and pd.notna(r[c]) and str(r[c]).strip():
            name_parts.append(str(r[c]).strip())
            break
    name = " ".join(name_parts) if name_parts else f"Customer {cid}"

    addr_parts = []
    for c in ["address", "street", "city", "state", "country", "postcode", "zip"]:
        if c in r and pd.notna(r[c]) and str(r[c]).strip():
            addr_parts.append(str(r[c]).strip())
    address = ", ".join(addr_parts)
    return name, address

# ---------------- STREAMLIT APP ----------------
def main():
    st.set_page_config(page_title="AI-Driven Personalization and Ethics in Banking", page_icon="💡", layout="wide")

    # Styling + Header
    st.markdown("""
    <style>
    [data-testid="stAppViewContainer"] > .main {
        background-color: #0E1117;
        color: #FAFAFA;
        font-family: "Helvetica Neue", sans-serif;
    }
    h1, h2, h3 { color: #00BFA5; }
    .stMetricValue, .stMetricLabel { color: #FAFAFA !important; }
    .small-note { color:#9aa0a6; font-size:0.9rem; }
    .big { font-size: 1.2rem; }
    </style>
    """, unsafe_allow_html=True)

    st.markdown("""
    <div style="text-align:center;">
        <img src="https://www.ul.ie/themes/custom/ul/logo.svg" width="200">
        <h1 style="margin-bottom:0;">AI-Driven Personalization and Ethics in Banking</h1>
        <p style="color:gray; margin-top:5px;"> Recommendations • Governance • Ethics</p>
    </div>
    <hr>
    """, unsafe_allow_html=True)

    # Load + prepare
    with st.spinner("Loading and preparing model..."):
        customers, transactions, assets, limit_prices, close_prices, tx = load_data()
        pos, R, user_ids, item_ids, u2i, i2i = prepare_interactions(tx)
        X_item = compute_content_features(assets, limit_prices, close_prices, item_ids)
        item_vecs, norms, sims_c = prepare_similarity(R, X_item)

    out = pd.DataFrame()  # populated when we have recs

    # Sidebar
    st.sidebar.header("⚙️ Settings")
    selected = st.sidebar.selectbox("CustomerID", sorted(customers["customerID"].unique()))
    alpha = st.sidebar.slider("Blend (α):", 0.0, 1.0, 0.6, 0.05)
    topn = st.sidebar.slider("Top-N Recommendations", 5, 20, 10, 1)
    f_sector = st.sidebar.multiselect("Filter by Sector", sorted(assets["sector"].dropna().unique()))
    f_industry = st.sidebar.multiselect("Filter by Industry", sorted(assets["industry"].dropna().unique()))
    st.sidebar.markdown("---")

    # 🔐 Consent (GDPR) toggle
    st.sidebar.markdown("### 🔐 Data Use & Consent")
    user_consented = st.sidebar.checkbox("I confirm lawful use/consent for personalization", value=True)

    # KPIs
    k1, k2, k3 = st.columns(3)
    k1.metric("Users w/ BUYs", f"{pos['customerID'].nunique():,}")
    k2.metric("Asset Universe", f"{len(item_ids):,}")
    k3_placeholder = k3.empty()

    # Tabs
    tab1, tab2, tab3, tab4 = st.tabs([
        "🏦 Recommendations",
        "📊 Transactions",
        "🏛 Governance",
        "🌱 Ethics"
    ])

    # --- RECOMMENDATIONS ---
    with tab1:
        if not user_consented:
            st.info("Enable consent in the sidebar to generate recommendations.")
        else:
            recs, reasons = recommend_for_user(
                selected, alpha, topn, customers, tx, user_ids, item_ids, u2i, i2i,
                R, item_vecs, norms, sims_c, assets, X_item
            )

            if len(recs)==0:
                st.warning("No recommendations found.")
            else:
                meta = (assets.drop_duplicates("ISIN")
                        .set_index("ISIN")[["assetName","assetCategory","sector","industry","marketID"]]
                        .reindex(recs).reset_index())
                perf = limit_prices.set_index("ISIN")[["profitability"]].reindex(recs).reset_index()
                out = meta.merge(perf, on="ISIN", how="left")
                out["reason"] = reasons[:len(out)]

                # User filters
                if f_sector: out = out[out["sector"].isin(f_sector)]
                if f_industry: out = out[out["industry"].isin(f_industry)]

                k3_placeholder.metric("Avg Profitability (Top-N)", f"{out['profitability'].mean():.2f}")

                st.subheader(f"Recommendations for Customer {selected}")
                st.data_editor(
                    out[["ISIN","assetName","assetCategory","sector","industry","marketID","profitability","reason"]],
                    hide_index=True, disabled=True, use_container_width=True
                )

                # ---- Profitability chart with angled, wrapped labels ----
                fig, ax = plt.subplots(figsize=(11, 5))
                labels = out["assetName"].fillna("N/A").apply(lambda s: _wrap_label(s, width=12, max_lines=2))
                heights = out["profitability"].fillna(0).astype(float)
                bars = ax.bar(labels, heights, color="#00BFA5", edgecolor="#0E1117", linewidth=0.5)
                for b in bars:
                    h = b.get_height()
                    ax.text(
                        b.get_x() + b.get_width()/2,
                        h + (0.05 if h >= 0 else -0.15),
                        f"{h:.0f}",
                        ha="center",
                        va="bottom" if h >= 0 else "top",
                        fontsize=9,
                        color="#222"
                    )
                ax.set_ylabel("Profitability", fontsize=11)
                ax.set_title("Profitability (Top-N)", fontsize=15, fontweight="bold", color="#00BFA5", pad=10)
                ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.4)
                plt.xticks(rotation=30, ha="right", fontsize=9)
                plt.yticks(fontsize=9)
                plt.tight_layout()
                st.pyplot(fig)

                # "Why" section with customer details when available
                with st.expander("💬 Why these were recommended?"):
                    cust_name, cust_addr = _customer_display(customers, selected)
                    where_bit = f" from {cust_addr}" if cust_addr else ""
                    avg_prof = float(out["profitability"].fillna(0).mean()) if not out.empty else 0.0

                    st.markdown(f"""
**Customer Profile:** {cust_name}{where_bit}
This customer has a history of investing in similar types of assets.

**Reason for Recommendation:**
- Based on previous purchases, these assets are the closest match to what the customer has shown interest in.
- The system also considers the customer’s **declared risk level**, ensuring recommendations remain within the bank’s suitability and compliance policy.
- By analysing behaviour of **similar customers** (collaborative filtering) and **features of assets** (content analysis), the model suggests products most relevant to the customer’s profile.
- The **average historical profitability** for these assets is approximately **{avg_prof:.2f}**, though past performance does not guarantee future returns.
- These recommendations are designed to **support advisors** in offering informed, risk-appropriate suggestions — not to make autonomous investment decisions.
                    """)

                st.download_button(
                    "⬇️ Download Recommendations (CSV)",
                    out.to_csv(index=False),
                    file_name=f"recommendations_{selected}.csv",
                    mime="text/csv"
                )
                st.caption("**Disclaimer:** This is an assistive prototype to support human decision-making; it is not investment advice.")

    # --- TRANSACTIONS ---
    with tab2:
        st.subheader("Recent BUY Transactions")
        recent = (
            tx[(tx["customerID"]==selected)&(tx["transactionType"].str.lower()=="buy")]
            .sort_values("timestamp",ascending=False)
            .head(20)[["timestamp","ISIN","assetName","assetCategory","totalValue","units"]]
        )
        st.data_editor(recent, hide_index=True, disabled=True, use_container_width=True)

        st.download_button(
                    "⬇️ Download Customer Statement (CSV)",
                    out.to_csv(index=False),
                    file_name=f"transactions_{selected}.csv",
                    mime="text/csv"
                )
        st.caption("Top 20 recent BUY transactions for this customer.")

    # --- GOVERNANCE ---
    with tab3:
        st.subheader("Policy & Compliance")
        policy_rows = {
            "Framework": [
                "GDPR (2016/2018)",
                "MiFID II (2018)",
                "Internal Risk–Suitability Policy",
                "EU AI Act (2024)",
                "Transparency / Explainability"
            ],
            "Requirement": [
                "Lawful basis / consent before personalization",
                "Suitability checks (product matches client risk)",
                "Enforce allowed categories per risk profile",
                "Risk documentation & human oversight for higher-risk AI",
                "Provide reasons users can understand"
            ],
            "Status": ["PASS", "PASS", "PASS", "INCOMPLETE", "PASS"],
            "Evidence / Notes": [
                "User consent confirmed in app",
                "Recommended categories ⊆ allowed for client's risk",
                "Policy mapping applied (risk filter active)",
                "Need documented risk classification / DPIA",
                "Plain-English reason provided with each recommendation"
            ]
        }
        st.data_editor(pd.DataFrame(policy_rows), hide_index=True, disabled=True, use_container_width=True)

        st.subheader("Sign-off")
        approver_df = pd.DataFrame({
            "Approver Name": ["Priya Menon", "Liam O’Connor", "Sara Novak", "Alan Murphy"],
            "Role": ["Model Risk Manager", "Compliance Officer", "Data Protection Officer (DPO)", "Head of Advisory"]
        })
        st.data_editor(approver_df, hide_index=True, disabled=True, use_container_width=True)

    # --- ETHICS (unchanged) ---
    with tab4:
        st.subheader("Responsible AI Framework & Stewardship Roles")
        st.caption("This framework outlines how Responsible AI values are implemented and overseen by designated data stewards and governance roles.")

        ethics_data = {
            "Principle": [
                "Fairness",
                "Transparency & Explainability",
                "Accountability",
                "Non-Maleficence (Do No Harm)",
                "Human Oversight",
                "Privacy & Data Protection"
            ],
            "Implementation": [
                "Risk-based filtering ensures customers only see suitable products.",
                "Each recommendation includes a plain-English reason.",
                "Fairness dashboard enables audit; logs/artefacts can be reviewed.",
                "Conservative users are protected from high-volatility categories.",
                "Human-in-the-loop via α blend, filters, Top-N; recommendations are assistive.",
                "Open, anonymized dataset used; if personal data introduced, follow DPIA & minimization."
            ],
            "Approver": [
                "Model Risk Manager",
                "AI Ethics Committee",
                "Compliance Officer",
                "Chief Risk Officer (CRO)",
                "Business Owner",
                "Data Protection Officer (DPO)"
            ]
        }
        st.data_editor(pd.DataFrame(ethics_data), hide_index=True, disabled=True, use_container_width=True)

        with st.expander("📝 Risk Checklist"):
            c1, c2 = st.columns(2)
            c1.checkbox("Data minimization documented")
            c1.checkbox("Purpose limitation documented")
            c1.checkbox("Data subject rights process (access/erasure)")
            c2.checkbox("Risk register updated")
            c2.checkbox("Human oversight defined")
            c2.checkbox("Mitigations documented")
            st.caption("These acknowledgements are demo controls; in production they’d be part of governance workflow.")

        # === Fairness Dashboard (with simple metrics) ===
        st.subheader("Fairness Risk Dashboard")

        tx_clean = tx.copy()
        tx_clean["riskLevel"] = (
            tx_clean["riskLevel"].astype(str).str.strip()
            .replace({"nan": np.nan, "None": np.nan, "": np.nan})
            .str.title()
        ).fillna("Not Provided")
        tx_clean["riskLevel"] = tx_clean["riskLevel"].replace({
            "Aggresive": "Aggressive",
            "Conservativ": "Conservative",
            "Income ": "Income",
        })
        risk_order = ["Conservative", "Income", "Balanced", "Aggressive", "Not Provided"]
        tx_clean["riskLevel"] = pd.Categorical(tx_clean["riskLevel"], categories=risk_order, ordered=True)

        fairness = (
            tx_clean.groupby("riskLevel", observed=False)
                    .agg(avg_value=("totalValue","mean"),
                         n=("totalValue","size"))
                    .reset_index()
                    .dropna(subset=["riskLevel"])
                    .sort_values("riskLevel")
        )

        hi_row = fairness.loc[fairness["avg_value"].idxmax()]
        lo_row = fairness.loc[fairness["avg_value"].idxmin()]
        den = lo_row["avg_value"] if lo_row["avg_value"] != 0 else 1e-9
        fairness_ratio = float(hi_row["avg_value"]) / float(den)

        c1, c2, c3 = st.columns(3)
        c1.metric("Highest avg (risk level)", f"{hi_row['riskLevel']}", f"{hi_row['avg_value']:,.0f}")
        c2.metric("Lowest avg (risk level)",  f"{lo_row['riskLevel']}", f"{lo_row['avg_value']:,.0f}")
        c3.metric("Fairness ratio (max/min)", f"{fairness_ratio:.2f}×",
                  delta=("OK" if fairness_ratio <= 2.0 else "Review"),
                  delta_color=("normal" if fairness_ratio <= 2.0 else "inverse"))

        fig_fair, ax_fair = plt.subplots(figsize=(10, 5))
        xlabels = fairness["riskLevel"].astype(str).apply(lambda s: _wrap_label(s, width=12, max_lines=2))
        bars = ax_fair.bar(xlabels, fairness["avg_value"], color="#4DD0E1", edgecolor="#0E1117", linewidth=0.5)
        for b in bars:
            h = b.get_height()
            ax_fair.text(b.get_x() + b.get_width()/2, h + max(0.02*h, 300),
                         f"{h:,.0f}", ha="center", va="bottom", fontsize=9, color="#222")
        ax_fair.set_title("Average Transaction Value by Risk Level", fontsize=15, fontweight="bold", color="#00BFA5", pad=10)
        ax_fair.set_xlabel("Risk Level", fontsize=11); ax_fair.set_ylabel("Average Total Value", fontsize=11)
        ax_fair.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.4)
        plt.xticks(rotation=25, ha="right", fontsize=10); plt.yticks(fontsize=10)
        plt.tight_layout()
        st.pyplot(fig_fair)

        st.markdown(
            f"""
### 🟢 Fairness Summary
- The **highest average** transaction value is for **{hi_row['riskLevel']}** customers: **€{hi_row['avg_value']:,.0f}** (n={int(hi_row['n'])}).
- The **lowest average** is for **{lo_row['riskLevel']}** customers: **€{lo_row['avg_value']:,.0f}** (n={int(lo_row['n'])}).
- The **difference ratio is {fairness_ratio:.2f}×**, which is **below the 2× check threshold** — **no fairness concern flagged** by this basic test.
- This is consistent with expectations (e.g., *Aggressive* users typically make larger trades than *Conservative* users).
- Treat this as an **early warning signal** only; for stronger assurance, also compare medians and control for confounders (income, tenure).
            """
        )

        st.download_button("⬇️ Download Audit Log (JSON)", data="[]", file_name="audit_log.json", mime="application/json")

    # Footer
    st.markdown("<hr><div style='text-align:center;color:gray'>© 2025 Rajan Raorane • MSc AI • University of Limerick</div>", unsafe_allow_html=True)

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'streamlit'